In [1]:
import sys
print(sys.executable)

c:\Users\Arsath Mohamed\AppData\Local\Programs\Python\Python312\python.exe


In [2]:
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3")

In [7]:
llm_local = ChatOpenAI(
    base_url="http://localhost:12434/v1", 
    api_key="not-needed",      
    model="llama3.1",      
    temperature=0.9      
)

In [8]:
response = llm_local.invoke("Explain transformers in 1 line")
print(response.content)

Transformers are electrical devices that transfer energy from one circuit to another through electromagnetic induction, changing voltage levels while maintaining current levels, to facilitate efficient power transmission and distribution.


In [9]:
from langchain_community.document_loaders import PyPDFLoader


loader = PyPDFLoader("./VectorDB/enterprise_sales_report.pdf")

docs = loader.load()

docs



incorrect startxref pointer(1)
parsing for Object Streams


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-04-28T18:19:03+05:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-04-28T18:19:03+05:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': './VectorDB/enterprise_sales_report.pdf', 'total_pages': 201, 'page': 0, 'page_label': '1'}, page_content='Global Sales Performance Report\nPrepared by: Analytics Team\nGenerated on: 1995-01-21'),
 Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-04-28T18:19:03+05:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-04-28T18:19:03+05:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': './VectorDB/enterprise_sales_report.pdf', 'total_pages': 201, 'page': 1, 'page_label': '2'}, page_content='Sales Report - Section 1\nExecutive Summary\nTheir baby institution remain. Tr

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100)

chunks = splitter.split_documents(docs)
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-04-28T18:19:03+05:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-04-28T18:19:03+05:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': './VectorDB/enterprise_sales_report.pdf', 'total_pages': 201, 'page': 0, 'page_label': '1'}, page_content='Global Sales Performance Report\nPrepared by: Analytics Team\nGenerated on: 1995-01-21'),
 Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-04-28T18:19:03+05:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-04-28T18:19:03+05:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': './VectorDB/enterprise_sales_report.pdf', 'total_pages': 201, 'page': 1, 'page_label': '2'}, page_content='Sales Report - Section 1\nExecutive Summary\nTheir baby institution remain. Tr

In [11]:
chunks[0].metadata

{'producer': 'ReportLab PDF Library - (opensource)',
 'creator': '(unspecified)',
 'creationdate': '2026-04-28T18:19:03+05:00',
 'author': '(anonymous)',
 'keywords': '',
 'moddate': '2026-04-28T18:19:03+05:00',
 'subject': '(unspecified)',
 'title': '(anonymous)',
 'trapped': '/False',
 'source': './VectorDB/enterprise_sales_report.pdf',
 'total_pages': 201,
 'page': 0,
 'page_label': '1'}

In [18]:
from langchain_ollama import OllamaEmbeddings

embedding_model = OllamaEmbeddings(
    model="nomic-embed-text"
)

In [20]:


from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)



In [21]:
vectorstore.similarity_search("Which region has the highest sales?", k= 3)

[Document(metadata={'subject': '(unspecified)', 'trapped': '/False', 'author': '(anonymous)', 'source': './VectorDB/enterprise_sales_report.pdf', 'creationdate': '2026-04-28T18:19:03+05:00', 'creator': '(unspecified)', 'keywords': '', 'title': '(anonymous)', 'total_pages': 201, 'page': 147, 'page_label': '148', 'moddate': '2026-04-28T18:19:03+05:00', 'producer': 'ReportLab PDF Library - (opensource)'}, page_content='Sales Report - Section 74\nExecutive Summary\nLight majority serious model sell. Accept issue situation such adult strong measure. Scene people born\neffect through. Drop would see image free it relate.\nKey Performance Indicators\nMetric\nValue\nTotal Revenue\n$212,921.00\nTotal Orders\n286\nAvg Order Value\n$744.48\nGrowth %\n2.64%\nDetailed Sales Data\nDate\nCustomer\nRegion\nProduct\nQty\nUnit Price\nRevenue\n2020-02-20\nMark Ellis\nIndiana\nWord\n8\n$436.83\n$3,494.64\n2005-06-28\nKristie Diaz\nUtah\nRoad\n1\n$317.82\n$317.82\n1997-12-21\nDanielle Strong\nMichigan\nReg

In [22]:
context = vectorstore.similarity_search("Which region has the highest sales?", k= 3)

In [23]:
response = llm.invoke(f"Which region has the highest sales? You can answer using the following context: {context}")
print(response.content)

A treasure trove of sales data!

After analyzing the documents, I found that the region with the highest sales is Utah.

Here's a breakdown of the top-selling regions:

1. **Utah**: Total Revenue - $3,812.45 (from 2 transactions)
	* 2020-02-20: Mark Ellis ($3,494.64)
	* 1980-08-02: Amy Dorsey ($813.63)

Other notable regions include:

2. **Indiana**: Total Revenue - $3,494.64 (from 1 transaction)
3. **Hawaii**: Total Revenue - $2,853.20 (from 1 transaction)
4. **Michigan**: Total Revenue - $1,366.65 (from 1 transaction)

Please note that these figures are based on the limited data provided in the documents and may not represent the complete sales picture for each region.

Would you like me to analyze any other aspects of the sales data or provide more insights?
